# SOP/PCOS Model Training - Overfitting Diagnosis & Regularization

This notebook retrains the SOP (PCOS) prediction model with overfitting diagnosis and regularization:

- **Overfitting Diagnosis**: Compare training accuracy vs 5-fold StratifiedKFold CV accuracy
- **Regularization**: Limit max_depth, increase min_samples_leaf, set max_features='sqrt'
- **GridSearch**: Find best hyperparameters that minimize train-CV gap
- **Feature Selection**: Optionally apply inside pipeline while accepting all 41 features

### Target Metrics
- Train-CV accuracy gap <= 0.05
- Test accuracy >= 0.92

### Pipeline Interface
The saved pipeline accepts all 41 features with exact column names (including spaces).
Structure: `Pipeline(StandardScaler + RandomForestClassifier)`

In [ ]:
!pip install scikit-learn joblib pandas numpy --quiet

In [ ]:
from google.colab import files

print("Please upload 'SOP_clean.csv'")
uploaded = files.upload()

# Get the filename from uploaded files
csv_filename = list(uploaded.keys())[0]
print(f"\nUploaded: {csv_filename} ({len(uploaded[csv_filename])} bytes)")

In [ ]:
import numpy as np
import pandas as pd

# Load the dataset
df = pd.read_csv(csv_filename)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}: {repr(col)}")

print(f"\nFirst 5 rows:")
display(df.head())

print(f"\nTarget distribution (PCOS (Y/N)):")
print(df['PCOS (Y/N)'].value_counts())
print(f"\nPositive rate: {df['PCOS (Y/N)'].mean():.3f}")

print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Dataset samples: {len(df)}, Features: {len(df.columns) - 1}")

In [ ]:
# Target column
TARGET = 'PCOS (Y/N)'

# All 41 feature names - must match EXACTLY what's in the CSV
# (including leading/trailing spaces)
y = df[TARGET]
X = df.drop(columns=[TARGET])

FEATURE_NAMES = list(X.columns)

print(f"Number of features: {len(FEATURE_NAMES)}")
print(f"Number of samples: {len(X)}")
print(f"Feature-to-sample ratio: {len(FEATURE_NAMES)/len(X):.3f}")
print(f"\nFeature names (with repr to show spaces):")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {i:2d}: {repr(name)}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

## Diagnose Overfitting

Train the current architecture (RF with n_estimators=100, class_weight='balanced', no depth limit)
and compare training accuracy vs 5-fold StratifiedKFold CV accuracy.

If the gap > 0.05, overfitting is confirmed.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Current architecture (no regularization)
current_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
    )),
])

# Fit on training data
current_pipeline.fit(X_train, y_train)

# Training accuracy
train_accuracy = current_pipeline.score(X_train, y_train)

# 5-fold StratifiedKFold CV accuracy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(current_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
cv_accuracy = cv_scores.mean()

# Compute gap
gap = train_accuracy - cv_accuracy

print("=" * 50)
print("OVERFITTING DIAGNOSIS")
print("=" * 50)
print(f"  Training accuracy:    {train_accuracy:.4f}")
print(f"  CV accuracy (5-fold): {cv_accuracy:.4f}")
print(f"  Gap (train - CV):     {gap:.4f}")
print(f"  CV fold scores:       {[f'{s:.4f}' for s in cv_scores]}")
print("=" * 50)

if gap > 0.05:
    print(f"\n\u2717 OVERFITTING CONFIRMED (gap {gap:.4f} > 0.05)")
    print("  Applying regularization...")
    overfitting_confirmed = True
else:
    print(f"\n\u2713 No significant overfitting (gap {gap:.4f} <= 0.05)")
    overfitting_confirmed = False

## Apply Regularization

Use GridSearchCV to find the best regularized hyperparameters:
- `max_depth`: [5, 6, 7, 8]
- `min_samples_leaf`: [5, 8, 10]
- `max_features`: ['sqrt', 0.5]
- `n_estimators`: [100, 150]

Goal: minimize train-CV gap while maintaining test accuracy >= 0.92.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the regularized pipeline
reg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
    )),
])

# Hyperparameter grid for regularization
param_grid = {
    'clf__n_estimators': [100, 150],
    'clf__max_depth': [5, 6, 7, 8],
    'clf__min_samples_leaf': [5, 8, 10],
    'clf__max_features': ['sqrt', 0.5],
}

print(f"Grid search space: {2 * 4 * 3 * 2} = {2*4*3*2} combinations")
print(f"With 5-fold CV: {2*4*3*2 * 5} total fits\n")

# GridSearchCV with StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    reg_pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

print(f"\nBest CV accuracy: {grid_search.best_score_:.4f}")
print(f"Best parameters: {grid_search.best_params_}")

## Select Best Regularized Model

From the grid search results, select the model that:
1. Has train-CV gap <= 0.05
2. Maintains test accuracy >= 0.92

If multiple candidates meet both criteria, pick the one with highest CV accuracy.

In [ ]:
import pandas as pd

# Analyze grid search results
results_df = pd.DataFrame(grid_search.cv_results_)

# Compute train-CV gap for each candidate
results_df['gap'] = results_df['mean_train_score'] - results_df['mean_test_score']

# Filter candidates with gap <= 0.05
valid_candidates = results_df[results_df['gap'] <= 0.05].copy()

print(f"Total candidates: {len(results_df)}")
print(f"Candidates with gap <= 0.05: {len(valid_candidates)}")

if len(valid_candidates) > 0:
    # Sort by CV accuracy (descending)
    valid_candidates = valid_candidates.sort_values('mean_test_score', ascending=False)
    best_idx = valid_candidates.index[0]
    
    print(f"\nTop 5 candidates (gap <= 0.05, sorted by CV accuracy):")
    cols = ['params', 'mean_train_score', 'mean_test_score', 'gap']
    display(valid_candidates[cols].head())
else:
    print("\nNo candidates with gap <= 0.05 found.")
    print("Using best overall model from grid search.")
    valid_candidates = results_df.sort_values('mean_test_score', ascending=False)
    best_idx = valid_candidates.index[0]

# Get the best model
best_params = results_df.loc[best_idx, 'params']
best_train_score = results_df.loc[best_idx, 'mean_train_score']
best_cv_score = results_df.loc[best_idx, 'mean_test_score']
best_gap = results_df.loc[best_idx, 'gap']

print(f"\nSelected model:")
print(f"  Parameters: {best_params}")
print(f"  Train accuracy: {best_train_score:.4f}")
print(f"  CV accuracy:    {best_cv_score:.4f}")
print(f"  Gap:            {best_gap:.4f}")

# Build the final pipeline with best parameters
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=best_params['clf__n_estimators'],
        max_depth=best_params['clf__max_depth'],
        min_samples_leaf=best_params['clf__min_samples_leaf'],
        max_features=best_params['clf__max_features'],
        class_weight='balanced',
        random_state=42,
    )),
])

# Fit on full training data
final_pipeline.fit(X_train, y_train)
print(f"\nFinal pipeline trained on {len(X_train)} samples.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predictions on test set
y_pred = final_pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"  Test accuracy: {test_accuracy:.4f}  (target: >= 0.92)")
print(f"  Test accuracy meets target: {'\u2713 PASS' if test_accuracy >= 0.92 else '\u2717 FAIL'}")
print("=" * 50)

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No PCOS (0)', 'PCOS (1)']))

In [ ]:
# Final verification of train-CV gap with the selected model
final_train_accuracy = final_pipeline.score(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
final_cv_scores = cross_val_score(final_pipeline, X_train, y_train, cv=cv, scoring='accuracy')
final_cv_accuracy = final_cv_scores.mean()

final_gap = final_train_accuracy - final_cv_accuracy

print("=" * 50)
print("FINAL OVERFITTING CHECK")
print("=" * 50)
print(f"  Training accuracy:    {final_train_accuracy:.4f}")
print(f"  CV accuracy (5-fold): {final_cv_accuracy:.4f}")
print(f"  Gap (train - CV):     {final_gap:.4f}")
print(f"  CV fold scores:       {[f'{s:.4f}' for s in final_cv_scores]}")
print("=" * 50)

gap_ok = final_gap <= 0.05
acc_ok = test_accuracy >= 0.92

print(f"\n  Gap <= 0.05:          {'\u2713 PASS' if gap_ok else '\u2717 FAIL'} ({final_gap:.4f})")
print(f"  Test accuracy >= 0.92: {'\u2713 PASS' if acc_ok else '\u2717 FAIL'} ({test_accuracy:.4f})")

if gap_ok and acc_ok:
    print(f"\n\u2713 ALL TARGETS MET - Model is ready to save.")
else:
    print(f"\n\u2717 TARGETS NOT MET - Review hyperparameters.")

In [ ]:
import os
import joblib

# Create models directory
os.makedirs('models', exist_ok=True)

MODEL_PATH = 'models/modelo_sop_rf.joblib'
joblib.dump(final_pipeline, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")
print(f"File size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

In [ ]:
from google.colab import files

files.download(MODEL_PATH)
print(f"\nDownloading {MODEL_PATH}...")
print("Place this file in: Izel_app/python/models/modelo_sop_rf.joblib")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Reload the saved model to verify it works from disk
loaded_model = joblib.load(MODEL_PATH)

print("Pipeline Interface Verification")
print("=" * 50)

# 1. Is a Pipeline instance
assert isinstance(loaded_model, Pipeline), f"Not a Pipeline! Got: {type(loaded_model)}"
print("\u2713 Model is a sklearn Pipeline instance")

# 2. Has StandardScaler step
scaler_found = False
for name, step in loaded_model.named_steps.items():
    if isinstance(step, StandardScaler):
        scaler_found = True
        break
assert scaler_found, "No StandardScaler step found!"
print("\u2713 Pipeline contains StandardScaler step")

# 3. Exposes feature_names_in_ with all 41 features
assert hasattr(loaded_model, 'feature_names_in_'), "No feature_names_in_ attribute!"
actual_names = list(loaded_model.feature_names_in_)
assert len(actual_names) == 41, f"Expected 41 features, got {len(actual_names)}"
assert actual_names == FEATURE_NAMES, (
    f"Feature names mismatch!\n"
    f"Expected: {FEATURE_NAMES}\n"
    f"Actual: {actual_names}"
)
print(f"\u2713 feature_names_in_ has all 41 features (exact match)")

# 4. predict() returns int labels
sample = X_test.iloc[[0]]
pred = loaded_model.predict(sample)
assert pred[0] in (0, 1), f"Invalid prediction: {pred[0]}"
print(f"\u2713 predict() returns valid label: {pred[0]}")

# 5. predict_proba() returns float probabilities
proba = loaded_model.predict_proba(sample)
assert proba.shape == (1, 2), f"Invalid proba shape: {proba.shape}"
assert 0.0 <= proba[0, 0] <= 1.0, f"Invalid probability: {proba[0, 0]}"
assert 0.0 <= proba[0, 1] <= 1.0, f"Invalid probability: {proba[0, 1]}"
assert abs(proba[0, 0] + proba[0, 1] - 1.0) < 1e-6, "Probabilities don't sum to 1"
print(f"\u2713 predict_proba() returns valid probabilities: {proba[0]}")

# 6. Accepts DataFrame with all 41 columns
sample_dict = {name: [0.0] for name in FEATURE_NAMES}
sample_df = pd.DataFrame(sample_dict)
pred2 = loaded_model.predict(sample_df)
assert pred2[0] in (0, 1)
print(f"\u2713 Accepts DataFrame with all 41 columns")

print("\n" + "=" * 50)
print("ALL INTERFACE CHECKS PASSED")
print("=" * 50)
print(f"\nPipeline steps: {list(loaded_model.named_steps.keys())}")
print(f"Classifier: {loaded_model.named_steps['clf'].__class__.__name__}")
print(f"Classifier params:")
clf = loaded_model.named_steps['clf']
print(f"  n_estimators: {clf.n_estimators}")
print(f"  max_depth: {clf.max_depth}")
print(f"  min_samples_leaf: {clf.min_samples_leaf}")
print(f"  max_features: {clf.max_features}")